# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook:** c16 — Timber Stock Dataset Generation  
**Author:** Jasper Cluistra  
**Last Updated:** 2026-05-17

---

### Goal
Generate the timber stock datasets used by the optimizer when assigning cross-sections to structural members. New stock is drawn from a parameterised catalogue covering standard softwood sections; reclaimed stock is sampled from a synthetic donor building.

### Inputs
- Stock generation parameters (`c16_generation_timber`)
- Representative beam length statistics (`representative_beam_statistics.json` from c12_15)

### Outputs
| File | Description |
|------|-------------|
| `new_timber_v2.csv` | New softwood stock only |
| `complete_timber_v2.csv` | New + reclaimed stock combined — used by the optimizer |
| `complete_timber_small.csv` | Small mixed subset for fast development runs |

# 1. Stock generation
Generates new and reclaimed stock, combines them, and saves the results to `config.TIMBER_STOCK_PATH`. Set `efficient = False` to use realistic (randomised) transport distances instead of the minimum-distance baseline.

In [ ]:
import pandas as pd
import config
from c16_generation_timber import generate_new_stock, generate_reclaimed_stock

efficient = True  # True = minimum transport distance; False = realistic randomised distances

# ── Donor building selection ──────────────────────────────────────────────────
# "A" — 3-storey Dutch residential (hardcoded spans: 1800–4500 mm)
# "B" — commercial/industrial (spans derived from structure's min/max length statistics,
#        biased toward longer members to cover the >4500 mm range)
DONOR_BUILDING = "A"

# ── Generate New Stock ────────────────────────────────────────────────────────
df_new_stock = generate_new_stock(efficient)
display(df_new_stock.head(3))

# ── Generate Reclaimed Stock ──────────────────────────────────────────────────
df_reclaimed_stock = generate_reclaimed_stock(donor_building=DONOR_BUILDING)
display(df_reclaimed_stock.head(3))

print(f"Donor building: {DONOR_BUILDING}")
print(f"RS elements generated: {len(df_reclaimed_stock)}")
print(f"RS length range: {df_reclaimed_stock['Length'].min():.0f} – {df_reclaimed_stock['Length'].max():.0f} mm")
print(f"RS mean length:  {df_reclaimed_stock['Length'].mean():.0f} mm")

# ── Combine Datasets ──────────────────────────────────────────────────────────
df_combined_stock = pd.concat([df_new_stock, df_reclaimed_stock], ignore_index=True)

# ── Save ──────────────────────────────────────────────────────────────────────
new_stock_dataset_filename = 'new_timber.csv'
df_new_stock.to_csv(config.TIMBER_STOCK_PATH / new_stock_dataset_filename, index=False, sep=';')
combined_dataset_filename = f'complete_timber_{DONOR_BUILDING}.csv'
df_combined_stock.to_csv(config.TIMBER_STOCK_PATH / combined_dataset_filename, index=False, sep=';')

print(f"\nNew stock dataset saved as '{new_stock_dataset_filename}'.")
print(f"Combined dataset saved as '{combined_dataset_filename}'.")



Generating catalog...
  primary=306 beam types, tail=115 beam types

Length source stats (new stock):
  primary      (p5–p95):  (1800, 2100, 2400, 2700, 3000, 3300, 3600, 3900, 4200)
  short tail   (min→p5):  (600, 900, 1200, 1500)
  long tail    (p95→max): (4500, 4800, 5100, 5400, 5700, 6000)

Generated lengths — tail_margin_mm=300
  Sections per tail length (graduated 6..17):
    short tail: {600: 6, 900: 10, 1200: 13, 1500: 17}
    long tail:  {4500: 17, 4800: 15, 5100: 13, 5400: 10, 5700: 8, 6000: 6}

New stock generated: 421 elements total
  primary=306, short_tail=46, long_tail=69

Elements per length:
    600 mm: 6
    900 mm: 10
    1200 mm: 13
    1500 mm: 17
    1800 mm: 34
    2100 mm: 34
    2400 mm: 34
    2700 mm: 34
    3000 mm: 34
    3300 mm: 34
    3600 mm: 34
    3900 mm: 34
    4200 mm: 34
    4500 mm: 17
    4800 mm: 15
    5100 mm: 13
    5400 mm: 10
    5700 mm: 8
    6000 mm: 6



,Member_ID,State,Length,Depth,Width,Length_Category,Availability_Probability,f_mk,f_tk,E_modulus_eff,E_modulus_005,f_vk,f_c0k,k_density,mean_density,Origin_Country,Transport_Dist,EmissionFactor
0,NS_00000,0,1800.0,100.0,38.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,319.90,0.1781
1,NS_00001,0,1800.0,100.0,50.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,274.95,0.1764
2,NS_00002,0,1800.0,100.0,63.0,primary,1.0,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,327.86,0.1705


Donor building raw pool: 132 elements across 3 floors
  floor_joist: 30 raw -> 23 survived
  long_rafter: 42 raw -> 32 survived
  mid_beam: 36 raw -> 30 survived
  short_purlin: 24 raw -> 19 survived
Total surviving elements: 104

Reclaimed stock summary:
  Total elements: 104
  Length mean:    3704.3 mm
  Length std:     1559.3 mm
  Length min:     985 mm
  Length max:     5401 mm
  Unique lengths: 48

  By role:
    floor_joist: 23 elements, 2964–2978 mm, section 70x190 mm
    long_rafter: 32 elements, 5390–5401 mm, section 130x270 mm
    mid_beam: 30 elements, 4174–4189 mm, section 110x230 mm
    short_purlin: 19 elements, 985–1000 mm, section 60x140 mm


,Member_ID,State,Length,Depth,Width,Donor_Role,Cut_Loss_mm,f_mk,f_tk,E_modulus_eff,E_modulus_005,f_vk,f_c0k,k_density,mean_density,Origin_Country,Transport_Dist,EmissionFactor
0,RS_00001,1,2977.0,190.0,70.0,floor_joist,1,18.0,11.0,9000.0,6000.0,2.0,18.0,320.0,380.0,Netherlands,240.0,0.1767
1,RS_00002,1,2971.0,190.0,70.0,floor_joist,7,18.0,11.0,9000.0,6000.0,2.0,18.0,320.0,380.0,Netherlands,108.0,0.0457
2,RS_00003,1,2978.0,190.0,70.0,floor_joist,0,18.0,11.0,9000.0,6000.0,2.0,18.0,320.0,380.0,Netherlands,78.0,0.1772


Donor building: B
RS elements generated: 104
RS length range: 985 – 5401 mm
RS mean length:  3704 mm

New stock dataset saved as 'new_timber.csv'.
Combined dataset saved as 'complete_timber_B.csv'.


## 2. Section combinations
Inspect all unique Depth × Width combinations present in the new stock catalogue. Uncomment to run.

In [5]:
# depth_width_combinations = (
#     df_new_stock[['Depth', 'Width']]
#     .drop_duplicates()
#     .sort_values(by=['Depth', 'Width'])
#     .reset_index(drop=True)
# )
#
# print(f"\n{'#':<6} {'D (mm)':<12} {'W (mm)':<12}")
# print("-" * 30)
# for idx, (_, row) in enumerate(depth_width_combinations.iterrows(), 1):
#     print(f"{idx:<6} {int(row['Depth']):<12} {int(row['Width']):<12}")
# print("-" * 30)
# print(f"Total combinations: {len(depth_width_combinations)}")

## 3. Development subset
Generates a small mixed stock file (`complete_timber_small.csv`) for fast development and testing runs in the optimizer. Adjust `total_elements` and `reclaimed_ratio` as needed.

In [6]:
import config
from c16_generation_timber import generate_mixed_stock_subset

df_small = generate_mixed_stock_subset(
    total_elements   = 20,
    reclaimed_ratio  = 0.3,
    random_state     = 38,
    allow_replacement = True,
)

display(df_small.head(5))

dataset_filename = f'complete_timber_small_{DONOR_BUILDING}.csv'
df_small.to_csv(config.TIMBER_STOCK_PATH / dataset_filename, index=False, sep=';')
print(f"Subset saved as '{dataset_filename}'  ({len(df_small)} elements).")


Generating catalog...
  primary=306 beam types, tail=115 beam types

Length source stats (new stock):
  primary      (p5–p95):  (1800, 2100, 2400, 2700, 3000, 3300, 3600, 3900, 4200)
  short tail   (min→p5):  (600, 900, 1200, 1500)
  long tail    (p95→max): (4500, 4800, 5100, 5400, 5700, 6000)

Generated lengths — tail_margin_mm=300
  Sections per tail length (graduated 6..17):
    short tail: {600: 6, 900: 10, 1200: 13, 1500: 17}
    long tail:  {4500: 17, 4800: 15, 5100: 13, 5400: 10, 5700: 8, 6000: 6}

New stock generated: 421 elements total
  primary=306, short_tail=46, long_tail=69

Elements per length:
    600 mm: 6
    900 mm: 10
    1200 mm: 13
    1500 mm: 17
    1800 mm: 34
    2100 mm: 34
    2400 mm: 34
    2700 mm: 34
    3000 mm: 34
    3300 mm: 34
    3600 mm: 34
    3900 mm: 34
    4200 mm: 34
    4500 mm: 17
    4800 mm: 15
    5100 mm: 13
    5400 mm: 10
    5700 mm: 8
    6000 mm: 6

Donor building raw pool: 132 elements across 3 floors
  edge_beam: 24 raw -> 20 sur

,Member_ID,State,Length,Depth,Width,Length_Category,Availability_Probability,f_mk,f_tk,E_modulus_eff,E_modulus_005,f_vk,f_c0k,k_density,mean_density,Origin_Country,Transport_Dist,EmissionFactor,Donor_Role,Cut_Loss_mm
0,NS_00161,0,3000.0,250.0,50.0,primary,1.00,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Baltic States,1440.36,0.1795,NaN,NaN
1,NS_00055,0,2100.0,225.0,38.0,primary,1.00,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Belgium,158.57,0.1791,NaN,NaN
2,NS_00133,0,2700.0,300.0,100.0,primary,1.00,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Spain & Portugal,2004.11,0.1749,NaN,NaN
3,NS_00313,0,900.0,150.0,50.0,short_tail,0.05,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,France,460.83,0.1760,NaN,NaN
4,NS_00070,0,2400.0,100.0,63.0,primary,1.00,24.0,14.0,11000.0,7400.0,2.5,21.0,350.0,420.0,Germany,280.25,0.1737,NaN,NaN


Subset saved as 'complete_timber_small_B.csv'  (20 elements).
